In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class VisionLibrary:
    def __init__(self):
        self.sift = cv2.SIFT_create()
        self.matcher = cv2.BFMatcher()

    def get_feature_matches(self, ref_img, scene_img):
        kp1, des1 = self.sift.detectAndCompute(cv2.cvtColor(ref_img, cv2.COLOR_BGR2GRAY), None)
        kp2, des2 = self.sift.detectAndCompute(cv2.cvtColor(scene_img, cv2.COLOR_BGR2GRAY), None)

        if des1 is None or des2 is None:
            return None

        matches = self.matcher.knnMatch(des1, des2, k=2)
        good_matches = [m for m, n in matches if m.distance < 0.7 * n.distance]

        if len(good_matches) >= 4:
            src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
            dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
            
            _, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
            matches_mask = mask.ravel().tolist()
        else:
            matches_mask = None

        return cv2.drawMatches(ref_img, kp1, scene_img, kp2, good_matches, None,
                               matchColor=(0, 255, 0),
                               singlePointColor=None,
                               matchesMask=matches_mask, 
                               flags=2)

if __name__ == "__main__":
    lib = VisionLibrary()
    
    img_ref = cv2.imread('ref.png')
    img_scene = cv2.imread('scene.png')

    if img_ref is not None and img_scene is not None:
        output = lib.get_feature_matches(img_ref, img_scene)
        if output is not None:
            cv2.imwrite('task6_output.png', output)
            print("Task 6 output saved as task6_output.png")
    else:
        print("Error: Could not find ref.png or scene.png")

Task 6 output saved as task6_output.png
